# Old Code

In [ ]:
# import ast
# import numpy as np
# import pandas as pd
# import scipy.sparse as sp
# from datetime import datetime, timedelta
# import itertools

# from sklearn import logger

# from pyspark.sql import SparkSession, functions as F
# from pyspark.sql.window import Window
# from pyspark.ml.recommendation import ALS
# from pyspark.mllib.evaluation import RankingMetrics

# from lightfm.data import Dataset
# from lightfm import LightFM
# from lightfm.evaluation import precision_at_k, recall_at_k

# import mlflow
# import mlflow.spark

# import mlflow

# spark = SparkSession.builder \
#     .appName("aprioriReco_tb") \
#     .config("spark.sql.shuffle.partitions", "4000") \
#     .config("spark.executor.memoryOverhead", "4g") \
#     .config("spark.sql.files.ignoreCorruptFiles", "true") \
#     .config("spark.sql.parquet.enableVectorizedReader", "false") \
#     .config("spark.sql.parquet.mergeSchema", "true") \
#     .getOrCreate()

In [ ]:
# from datetime import datetime, timedelta
# from pyspark.sql import functions as F

# date = "2026-03-28"
# train_days = 90
# test_days = 30

# date_format = "%Y-%m-%d"
# base_date = datetime.strptime(date, date_format)

In [ ]:
# train_last_date = (base_date - timedelta(days=test_days)).strftime(date_format)
# apriori_results_path = f"gs://wynk-ml-workspace/xstream/clickApriori/day={train_last_date}/"
# print(f"Reading Apriori results from: {apriori_results_path}")
# apriori_df = spark.read.parquet(apriori_results_path)

Reading Apriori results from: gs://wynk-ml-workspace/xstream/clickApriori/day=2026-02-26/


In [ ]:
# from datetime import datetime, timedelta
# from pyspark.sql.utils import AnalysisException

# class DataPipeLine:
#     def __init__(self, spark, config):
#         self.spark = spark
#         self.config = config
#         self.train_df = None
#         self.test_df = None
#         self.apriori_results_df = None
#         self.metadata_df = None
#         self.indexed_train = None
#         self.indexed_test = None
#     def _get_valid_paths(self, base_path, start_days_ago, end_days_ago):
#         # Ensure base_date is parsed inside the method or passed in
#         base_date = datetime.strptime(self.config['date'], "%Y-%m-%d")
#         paths = []
#         for i in range(start_days_ago, end_days_ago):
#             target_date = (base_date - timedelta(days=i)).strftime("%Y-%m-%d")
#             paths.append(f"{base_path}/day={target_date}")
#         return paths

#     def _read_and_union_paths(self, paths):
#         df = None
#         for path in paths:
#             try:
#                 temp_df = self.spark.read.parquet(path)
#                 df = temp_df if df is None else df.unionByName(temp_df)
#             except AnalysisException as e:
#                 print(f"Path not found or inaccessible, skipping: {path}")
#             except Exception as e:
#                 print(f"Unexpected error reading {path}: {e}")
#                 raise e
#         return df

#     def load_raw_data(self):
#         print("Loading raw interactions and metadata...")
        
#         # Test: Days 0 to test_days-1
#         test_paths = self._get_valid_paths(self.config['watch_history_path'], 0, self.config['test_days'])
        
#         # Train: Days test_days to (test_days + train_days - 1)
#         train_paths = self._get_valid_paths(self.config['watch_history_path'], self.config['test_days'], self.config['test_days'] + self.config['train_days'])        
        
#         # Apriori: Exactly Day 'test_days' (The bridge day)
#         apriori_path = self._get_valid_paths(self.config['apriori_results_path'], self.config['test_days'], self.config['test_days'] + 1)
        
#         self.test_df = self._read_and_union_paths(test_paths).select("userId").filter(F.col("userId").isNotNull() & (F.trim(F.col("userId")) != "")).distinct()
#         self.train_df = self._read_and_union_paths(train_paths).select("userId").filter(F.col("userId").isNotNull() & (F.trim(F.col("userId")) != "")).distinct()
#         self.apriori_results_df = self._read_and_union_paths(apriori_path)

        
#         # Load Metadata
#         tv_df = self.spark.read.parquet(f"{self.config['db_path']}{self.config['date']}/enriched_tv.parquet")
#         movie_df = self.spark.read.parquet(f"{self.config['db_path']}{self.config['date']}/enriched_movie.parquet")
        
#         def clean_meta(df):
#             return (df.filter((F.col('XstreamContentIds') != F.array()) & (F.col("published") == True))
#                     .withColumn("item_id_exploded", F.explode("XstreamContentIds")) # Step 1: Explode
#                     .select(
#                         F.col("item_id_exploded").cast("string").alias("item_id"), # Step 2: Cast
#                         "title", 
#                         F.col('OriginalLanguage').alias('original_language').cast("string"),
#                         "Genres"
#                     ))
#         self.metadata_df = clean_meta(tv_df).unionByName(clean_meta(movie_df)).distinct()

#         # Validation Check: Ensure we actually loaded something
#         if not self.train_df or not self.test_df or not self.apriori_results_df or not self.metadata_df:
#             raise ValueError("Critical Error: One or more DataFrames are empty/None. Check your paths!")

#     def process_and_index_data(self):
#         print("Filtering users, aggregating playtime, and building indexes...")
        
#         # 1. Overlapping Users & Aggregation
#         common_users = self.train_df.select("userId").distinct().join(self.test_df.select("userId").distinct(), "userId")
        
#         train_filtered = self.train_df.join(common_users, "userId")
#         test_filtered = self.test_df.join(common_users, "userId")
        
#         train_stats = train_filtered.groupBy("userId", "item_id") \
#             .agg(F.sum("total_play_time_sec").alias("total_playtime_combined")) \
#             .withColumn("distinct_content_count", F.count("item_id").over(Window.partitionBy("userId"))) \
#             .filter(F.col("total_playtime_combined").isNotNull() & ~F.isnan("total_playtime_combined"))
            
#         # --- NEW 95TH PERCENTILE LOGIC ---
#         # Get one row per user to ensure the percentile isn't skewed by row inflation
#         user_content_counts = train_stats.select("userId", "distinct_content_count").distinct()
        
#         # Calculate the 95th percentile
#         p95_threshold = user_content_counts.stat.approxQuantile("distinct_content_count", [0.95], 0.001)[0]
#         print(f"95th Percentile Threshold for max items watched: {p95_threshold}")
        
#         # Apply both the minimum (>= 10) and the maximum (<= p95) filters
#         als_input_base = train_stats.filter(
#             (F.col("distinct_content_count") >= 10) & 
#             (F.col("distinct_content_count") <= p95_threshold)
#         )
#         # ---------------------------------
        
#         # Re-filter test users based on the valid base
#         valid_users = als_input_base.select("userId").distinct()
#         test_filtered = test_filtered.join(valid_users, "userId")
        
#         # 2. Build Lookup Tables
#         distinct_users = valid_users.rdd.zipWithIndex().toDF(["user_struct", "userIndex"]).select(F.col("user_struct.*"), F.col("userIndex").cast("int"))
#         distinct_items = als_input_base.select("item_id").distinct().rdd.zipWithIndex().toDF(["item_struct", "itemIndex"]).select(F.col("item_struct.*"), F.col("itemIndex").cast("int"))
        
#         # Break Lineage!
#         distinct_users.write.mode("overwrite").parquet(f"{self.config['temp_path']}/user_lookup")
#         distinct_items.write.mode("overwrite").parquet(f"{self.config['temp_path']}/item_lookup")
        
#         user_lookup = self.spark.read.parquet(f"{self.config['temp_path']}/user_lookup")
#         self.item_lookup = self.spark.read.parquet(f"{self.config['temp_path']}/item_lookup")

#         # 3. Apply Indexes
#         self.indexed_train = als_input_base.join(user_lookup, "userId").join(self.item_lookup, "item_id") \
#             .withColumn("playtime_logged", F.log1p("total_playtime_combined")) \
#             .select("userIndex", "itemIndex", "playtime_logged").repartition(1000).cache()
            
#         self.indexed_test = test_filtered.join(user_lookup, "userId").join(self.item_lookup, "item_id") \
#             .withColumn("playtime_logged", F.log1p("total_play_time_sec")) \
#             .select("userIndex", "itemIndex", "playtime_logged").cache()

In [ ]:
# # 2. Get unique users (this is a heavy operation, so we do it once)
# # Using .distinct() on 500GB will trigger a shuffle.
# train_users = train_df.select("userId").filter(
#     F.col("userId").isNotNull() & 
#     (F.trim(F.col("userId")) != "")
# ).distinct()
# test_users = test_df.select("userId").filter(
#     F.col("userId").isNotNull() & 
#     (F.trim(F.col("userId")) != "")
# ).distinct()

# # 3. Join them to find the 'overlap'
# common_users = train_users.join(test_users, on="userId", how="inner")

# # 4. Filter the main dataframes
# # We use an inner join here because it's generally more 
# # efficient than 'isin' in Spark for large datasets.
# df_train_filtered = train_df.join(common_users, on="userId", how="inner")
# df_test_filtered = test_df.join(common_users, on="userId", how="inner")



In [ ]:
# enriched_db_path = "gs://wynk-ml-workspace/projects/xstream_nlu/catalog-db/" 

# enriched_tv_df = spark.read.parquet(f'{enriched_db_path}{date}/enriched_tv.parquet')
# enriched_movies_df = spark.read.parquet(f'{enriched_db_path}{date}/enriched_movie.parquet')

# filtered_enriched_tv = (
#     enriched_tv_df
#     # 1. Remove empty arrays and un-published content
#     .filter((F.col('XstreamContentIds') != F.array()) & (F.col("published") == True))
#     # 2. Explode and select in one go
#     .select(
#         F.explode("XstreamContentIds").alias("item_id"), # This is your primary key for ALS joins
#         "title", 
#         "ID",
#         F.col('OriginalLanguage').alias('original_language'), 
#         "Genres"
#     )
# )
# filtered_enriched_movies = (
#     enriched_movies_df
#     # 1. Remove empty arrays and un-published content
#     .filter((F.col('XstreamContentIds') != F.array()) & (F.col("published") == True))
#     # 2. Explode and select in one go
#     .select(
#         F.explode("XstreamContentIds").alias("item_id"), # This is your primary key for ALS joins
#         "title", 
#         "ID", 
#         F.col('OriginalLanguage').alias('original_language'),
#         "Genres"
#     )
# )

# # 1. Ignore files that are physically broken/incomplete
# spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")

# # 2. Disable the Vectorized Reader (Standardizes the reading process)
# spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")

# # 3. Handle schema mismatches if they exist
# spark.conf.set("spark.sql.parquet.mergeSchema", "true")

# # 4. Perform the union with explicit casting to ensure types match
# tv_final = filtered_enriched_tv.select(
#     F.col("item_id").cast("string"),
#     F.col("title").cast("string"),
#     F.col("original_language").cast("string"),
#     F.col("Genres").cast("string") # Ensure both are cast to the same type
# )

# movies_final = filtered_enriched_movies.select(
#     F.col("item_id").cast("string"),
#     F.col("title").cast("string"),
#     F.col("original_language").cast("string"),
#     F.col("Genres").cast("string")
# )

# enriched_content_df = tv_final.unionByName(movies_final).distinct()
# # enriched_content_df.cache().count() # Use count() to force Spark to read and verify all files now
# enriched_content_df = filtered_enriched_tv.unionByName(filtered_enriched_movies)


In [ ]:
# def compute_user_stats(watch_history_df):
#     user_stats = (
#         watch_history_df  
#         .groupBy("userId", "item_id")
#         .agg(
#             F.sum("total_play_time_sec").alias("total_playtime_combined")
#         )
#         # We add a column to count how many distinct items EACH user has watched
#         .withColumn("distinct_content_count", F.count("item_id").over(Window.partitionBy("userId")))
#         .filter(
#     F.col("total_playtime_combined").isNotNull() & 
#     ~F.isnan("total_playtime_combined")
# )
#     )
#     return user_stats

# train_combined_stats = compute_user_stats(df_train_filtered)

# # 2. Filter for users with at least 3 distinct items
# als_input_base = train_combined_stats.filter("distinct_content_count >= 2")

# # als_input_base.cache()
# # 3. Join them to find the 'overlap'
# re_common_users = test_users.join(als_input_base, on="userId", how="inner").cache()

# # 4. Filter the main dataframes
# # We use an inner join here because it's generally more 
# # efficient than 'isin' in Spark for large datasets.
# df_test_filtered = df_test_filtered.join(re_common_users, on="userId", how="inner")

we have user watch history with trained_combined_stats. Use this watch history to generate apriori recos for each user? like get all simlar items for each item, combine them all so that the common ones get more weightage, and get top 15 using the similarity score. 

In [ ]:
# apriori_df.show(5, truncate=False)

+----------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
# import pyspark.sql.functions as F

# # 1. Explode the array first
# exploded_df = apriori_df.select(
#     "item_id",
#     F.explode("similar_items").alias("sim_item_obj")
# )

# # 2. Extract and explicitly alias the struct fields
# clean_exploded_sim_df = exploded_df.select(
#     F.col("item_id").alias("item_id"), # The original watched item
#     F.col("sim_item_obj.item_id").alias("rec_item_id"), # Rename the struct's item_id
#     F.col("sim_item_obj.score").alias("score")
# )

# # clean_exploded_sim_df.show(5, truncate=False)
# # Optional check to verify your schema is clean
# # clean_exploded_sim_df.printSchema()

In [ ]:
# user_candidates = als_input_base.join(clean_exploded_sim_df, on="item_id", how="inner")
# user_candidates.show(10, truncate=False)

26/03/30 09:32:16 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/30 09:32:31 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 63 for reason Executor for container container_1764236692086_5990_01_000081 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/30 09:32:31 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 64 for reason Executor for container container_1764236692086_5990_01_000083 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/30 09:32:31 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 55 for reason Executor for container container_1764236692086_5990_01_000069 exited because of

+-----------------------------------+------------------+-----------------------+----------------------+-----------------------------------------------------+------------+
|item_id                            |userId            |total_playtime_combined|distinct_content_count|rec_item_id                                          |score       |
+-----------------------------------+------------------+-----------------------+----------------------+-----------------------------------------------------+------------+
|LIONSGATEPLAY_MOVIE_JOHNRAMBOY2008M|-07LdbjtwoQx3h0vK0|48.0                   |12                    |LIONSGATEPLAY_MOVIE_ANGELSFALLENWARRIORSOFPEACEY2024M|2.2389857E-7|
|LIONSGATEPLAY_MOVIE_JOHNRAMBOY2008M|-07LdbjtwoQx3h0vK0|48.0                   |12                    |LIONSGATEPLAY_MOVIE_SNITCHY2013M                     |2.545086E-7 |
|LIONSGATEPLAY_MOVIE_JOHNRAMBOY2008M|-07LdbjtwoQx3h0vK0|48.0                   |12                    |LIONSGATEPLAY_MOVIE_FURYY2014M            

In [ ]:
# user_aggregated_scores = user_candidates.groupBy("userId", "rec_item_id") \
#     .agg(F.sum("score").alias("total_score"))

# user_aggregated_scores.show(10, truncate=False)

26/03/30 09:32:48 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/30 09:32:56 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


+------------------+------------------------------------------------------+---------------------+
|userId            |rec_item_id                                           |total_score          |
+------------------+------------------------------------------------------+---------------------+
|-07LdbjtwoQx3h0vK0|CHAUPAL_MOVIE_en_9ac2d568-3de7-4e10-995b-d960fbdc4b36 |1.5519625549131888E-6|
|-07LdbjtwoQx3h0vK0|CHAUPAL_MOVIE_en_8b879bae-8e37-4321-b404-458e7d8e37f1 |9.901424618874444E-6 |
|-07LdbjtwoQx3h0vK0|CHAUPAL_MOVIE_en_movie_860                            |3.293372787993576E-5 |
|-07LdbjtwoQx3h0vK0|CHAUPAL_MOVIE_en_3f4da6e4-dcc9-4e2e-8e5e-53b1131821af |1.6564950101383147E-5|
|-07LdbjtwoQx3h0vK0|CHAUPAL_MOVIE_en_f8220388-c37d-4b40-b898-6bfc94987614 |3.3389763984814635E-6|
|-07LdbjtwoQx3h0vK0|CHAUPAL_MOVIE_en_2eace4b0-13fe-4716-af1d-f6dde4aae6cf |1.5546390841336688E-5|
|-07LdbjtwoQx3h0vK0|CHAUPAL_MOVIE_en_movie_766                            |2.5232287839571654E-5|
|-07LdbjtwoQx3h0vK0|

In [ ]:
# watched_items = train_combined_stats.selectExpr("userId", "item_id as rec_item_id")

# new_recommendations = user_aggregated_scores.join(
#     watched_items,
#     on=["userId", "rec_item_id"],
#     how="left_anti"
# )

# new_recommendations.show(20, truncate=False)

26/03/30 09:33:01 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/30 09:33:08 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


+------------------+--------------------------------------------------------------+---------------------+
|userId            |rec_item_id                                                   |total_score          |
+------------------+--------------------------------------------------------------+---------------------+
|---OTo5hecghK8HZ00|ZEEFIVE_MOVIE_0-0-sanamterikasam                              |1.9859324311255477E-5|
|--03-VasNdpfk0Fa20|MINITV_MOVIE_amzn1.dv.gti.80a47849-3ae9-4dcf-b3c9-0d4f45112b59|6.72970088544389E-8  |
|--03-VasNdpfk0Fa20|ZEEFIVE_MOVIE_0-0-1z5535878                                   |9.281721759180073E-6 |
|--0BIkmJnxqdVyagC0|MINITV_MOVIE_amzn1.dv.gti.cf4d03cf-d717-47e8-ace7-bc79e82003ae|1.917370298087917E-7 |
|--17p0Yk5D3V9JME40|SONYLIV_VOD_MOVIE_1090478869                                  |4.451972301922069E-7 |
|--1DTm2Q_9jEIB-Is0|SONYLIV_VOD_MOVIE_1000223488                                  |1.4020764105282524E-8|
|--1PST1UEtWRcyWFd0|ZEEFIVE_TVSHOW_0-6-4z58529

26/03/30 09:33:46 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 27 for reason Executor for container container_1764236692086_5990_01_000030 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/30 09:34:52 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 101 for reason Executor for container container_1764236692086_5990_01_000125 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/30 09:34:52 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 102 for reason Executor for container container_1764236692086_5990_01_000126 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/30 09:34:52 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 90 for reason Executor for container container_1764236692086

# TestBench

In [ ]:
'daily_click_watch_history_save_path': 'gs://wynk-ml-workspace/projects/rails_reranking/daily_user_click_watch_history_new'

# Cleaner Code

To reduce spaghetti, I'll create functions and maybe proceed with classes to reduce clutter.

In [1]:
import ast
import numpy as np
import pandas as pd
import scipy.sparse as sp
from datetime import datetime, timedelta
import itertools

from sklearn import logger

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark.ml.recommendation import ALS
from pyspark.mllib.evaluation import RankingMetrics

from lightfm.data import Dataset
from lightfm import LightFM
from lightfm.evaluation import precision_at_k, recall_at_k

import mlflow
import mlflow.spark

import mlflow

spark = SparkSession.builder \
    .appName("aprioriReco_tb") \
    .config("spark.sql.shuffle.partitions", "4000") \
    .config("spark.executor.memoryOverhead", "4g") \
    .config("spark.sql.files.ignoreCorruptFiles", "true") \
    .config("spark.sql.parquet.enableVectorizedReader", "false") \
    .config("spark.sql.parquet.mergeSchema", "true") \
    .getOrCreate()

bash: /opt/conda/miniconda3/lib/libtinfo.so.6: no version information available (required by bash)
bash: /opt/conda/miniconda3/lib/libtinfo.so.6: no version information available (required by bash)
26/04/01 06:30:58 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/04/01 06:30:58 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/01 06:30:58 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Pleas

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql import Window
from datetime import datetime, timedelta
from pyspark.sql.utils import AnalysisException

class DataPipeline:
    def __init__(self, spark, config):
        self.spark = spark
        self.config = config
        
        # Raw DataFrames
        self.train_df = None
        self.test_df = None
        self.apriori_results_df = None
        self.metadata_df = None
        
        # Output DataFrames
        self.indexed_train = None
        self.indexed_test = None
        self.indexed_apriori = None 
        self.ground_data = None
        self.item_lookup = None

    def _get_valid_paths(self, base_path, start_days_ago, end_days_ago):
        """Generates partition paths and tracks days_ago for time decay."""
        base_date = datetime.strptime(self.config['date'], "%Y-%m-%d")
        paths_with_metadata = []
        for i in range(start_days_ago, end_days_ago):
            target_date = (base_date - timedelta(days=i)).strftime("%Y-%m-%d")
            paths_with_metadata.append({
                "path": f"{base_path}/day={target_date}", 
                "days_ago": i
            })
        return paths_with_metadata

    def _read_valid_paths_flat(self, path_dicts):
        """Validates paths on the driver and reads them all at once to keep the DAG flat."""
        valid_paths = []
        for p in path_dicts:
            path_string = p["path"]
            try:
                # Fast check: reading just the schema validates existence without loading data
                self.spark.read.parquet(path_string).schema
                valid_paths.append(path_string)
            except AnalysisException:
                print(f"Path not found or inaccessible, skipping: {path_string}")
            except Exception as e:
                print(f"Unexpected error validating {path_string}: {e}")
        
        if not valid_paths:
            print("Warning: No valid paths found in the provided range.")
            return None
            
        # A single, flat read of all confirmed-existing paths! No DAG explosion.
        return self.spark.read.parquet(*valid_paths)

    def load_raw_data(self):
        print("Loading raw interactions and metadata...")
        
        # 1. Generate path configurations
        test_path_dicts = self._get_valid_paths(self.config['watch_history_path'], 0, self.config['test_days'])
        train_path_dicts = self._get_valid_paths(self.config['watch_history_path'], self.config['test_days'], self.config['test_days'] + self.config['train_days'])        
        apriori_path_dicts = self._get_valid_paths(self.config['apriori_results_path'], self.config['test_days'], self.config['test_days'] + 1)
        
        # 2. Load and filter Test data natively
        raw_test = self._read_valid_paths_flat(test_path_dicts)
        if raw_test:
            self.test_df = raw_test.filter(F.col("userId").isNotNull() & (F.trim(F.col("userId")) != ""))
            
        # 3. Load and filter Train data natively
        raw_train = self._read_valid_paths_flat(train_path_dicts)
        if raw_train:
            self.train_df = raw_train.filter(F.col("userId").isNotNull() & (F.trim(F.col("userId")) != ""))
            
        # 4. Load Apriori Results
        self.apriori_results_df = self._read_valid_paths_flat(apriori_path_dicts)

        # 5. Load and Clean Metadata
        tv_df = self.spark.read.parquet(f"{self.config['db_path']}{self.config['date']}/enriched_tv.parquet")
        movie_df = self.spark.read.parquet(f"{self.config['db_path']}{self.config['date']}/enriched_movie.parquet")
        
        def clean_meta(df):
            return (df.filter((F.col('XstreamContentIds') != F.array()) & (F.col("published") == True))
                    .withColumn("item_id_exploded", F.explode("XstreamContentIds"))
                    .select(
                        F.col("item_id_exploded").cast("string").alias("item_id"),
                        "title", 
                        F.col('OriginalLanguage').alias('original_language').cast("string"),
                        "Genres"
                    ))
                    
        self.metadata_df = clean_meta(tv_df).unionByName(clean_meta(movie_df)).distinct()

        if not self.train_df or not self.test_df or not self.apriori_results_df or not self.metadata_df:
            raise ValueError("Critical Error: One or more DataFrames are empty/None. Check your paths!")

    def process_and_index_data(self):
        print("Filtering users, aggregating playtime, and building indexes...")
        
        # --- 0. Calculate Time Decay Natively ---
        base_date = F.lit(self.config['date']).cast("date")
        train_window = self.config['train_days'] - 1
        min_days = self.config['test_days'] 
        
        # Extract date from the file path and calculate days_ago
        train_with_date = self.train_df.withColumn(
            "partition_date", 
            F.regexp_extract(F.input_file_name(), r"day=(\d{4}-\d{2}-\d{2})", 1).cast("date")
        )
        
        train_with_decay = train_with_date.withColumn(
            "days_ago", F.datediff(base_date, F.col("partition_date"))
        ).withColumn(
            "decay_factor",
            1.0 - (0.5 * ((F.col("days_ago") - min_days) / max(1, train_window)))
        ).withColumn(
            "decayed_play_time",
            F.col("total_play_time_sec") * F.col("decay_factor")
        )

       # --- 1. Overlapping Users & Aggregation ---
        common_users = train_with_decay.select("userId").distinct().join(self.test_df.select("userId").distinct(), "userId")
        
        train_filtered = train_with_decay.join(common_users, "userId")
        test_filtered = self.test_df.join(common_users, "userId")
        
        # Group by user and item, summing the decayed play time AND keeping the most recent decay factor
        train_stats = train_filtered.groupBy("userId", "item_id") \
            .agg(
                F.sum("decayed_play_time").alias("total_playtime_combined"),
                F.max("decay_factor").alias("recent_decay_factor") # NEW: Capture the highest decay weight
            ) \
            .withColumn("distinct_content_count", F.count("item_id").over(Window.partitionBy("userId"))) \
            .filter(F.col("total_playtime_combined").isNotNull() & ~F.isnan("total_playtime_combined"))
            
        # Apply 95th Percentile Logic
        user_content_counts = train_stats.select("userId", "distinct_content_count").distinct()
        p95_threshold = user_content_counts.stat.approxQuantile("distinct_content_count", [0.95], 0.001)[0]
        print(f"95th Percentile Threshold for max items watched: {p95_threshold}")
        
        als_input_base = train_stats.filter(
            (F.col("distinct_content_count") >= self.config["distinct_user_content_threshold"]) & 
            (F.col("distinct_content_count") <= p95_threshold)
        )
        
        valid_users = als_input_base.select("userId").distinct()
        test_filtered = test_filtered.join(valid_users, "userId")
        
        # --- 2. Build Lookup Tables ---
        distinct_users = valid_users.rdd.zipWithIndex().toDF(["user_struct", "userIndex"]).select(F.col("user_struct.*"), F.col("userIndex").cast("int"))
        distinct_items = als_input_base.select("item_id").distinct().rdd.zipWithIndex().toDF(["item_struct", "itemIndex"]).select(F.col("item_struct.*"), F.col("itemIndex").cast("int"))
        
        distinct_users.write.mode("overwrite").parquet(f"{self.config['temp_path']}/user_lookup")
        distinct_items.write.mode("overwrite").parquet(f"{self.config['temp_path']}/item_lookup")
        
        user_lookup = self.spark.read.parquet(f"{self.config['temp_path']}/user_lookup")
        self.item_lookup = self.spark.read.parquet(f"{self.config['temp_path']}/item_lookup")

        # --- 3. Apply Indexes for ALS ---
        self.indexed_train = als_input_base.join(user_lookup, "userId").join(self.item_lookup, "item_id") \
            .withColumn("playtime_logged", F.log1p("total_playtime_combined")) \
            .select("userIndex", "itemIndex", "playtime_logged").repartition(1000).cache()
            
        self.indexed_test = test_filtered.join(user_lookup, "userId").join(self.item_lookup, "item_id") \
            .withColumn("playtime_logged", F.log1p("total_play_time_sec")) \
            .select("userIndex", "itemIndex", "playtime_logged").cache()

        # --- 4. Process Apriori Recommendations ---
        print("Processing Apriori Recommendations...")
        exploded_df = self.apriori_results_df.select(
            "item_id",
            F.explode("similar_items").alias("sim_item_obj")
        )

        clean_exploded_sim_df = exploded_df.select(
            F.col("item_id").alias("item_id"), 
            F.col("sim_item_obj.item_id").alias("rec_item_id"), 
            F.col("sim_item_obj.score").alias("raw_score") # Rename to raw_score
        )

        # Generate candidates using the filtered training base
        user_candidates = als_input_base.join(clean_exploded_sim_df, on="item_id", how="inner")

        # NEW: Apply the time decay to the similarity score
        user_candidates = user_candidates.withColumn(
            "decayed_sim_score", 
            F.col("raw_score") * F.col("recent_decay_factor")
        )

        # Aggregate using the newly decayed score
        user_aggregated_scores = user_candidates.groupBy("userId", "rec_item_id") \
            .agg(F.sum("decayed_sim_score").alias("total_score"))
            
        watched_items = als_input_base.selectExpr("userId", "item_id as rec_item_id")

        new_recommendations = user_aggregated_scores.join(
            watched_items,
            on=["userId", "rec_item_id"],
            how="left_anti"
        )
        
        # Index the Apriori output to match ALS
        self.indexed_apriori = new_recommendations \
            .join(user_lookup, "userId") \
            .join(self.item_lookup, F.col("rec_item_id") == F.col("item_id")) \
            .select("userIndex", F.col("itemIndex").alias("itemIndex"), "total_score").cache()

        # --- 5. Generate Ground Truth ---
        print("Generating Ground Truth...")
        self.ground_data = self.indexed_test.groupBy("userIndex").agg(F.collect_set("itemIndex").alias("actual_items")).cache()

    def get_als_data(self):
        """Returns DataFrames optimized for Spark ALS."""
        return self.indexed_train, self.ground_data
        
    def get_apriori_data(self):
        """Returns the processed Apriori recommendations."""
        return self.indexed_apriori
        
    def get_apriori_data_verbose(self):
        """Returns Apriori recommendations with human-readable userId and title."""
        user_lookup = self.spark.read.parquet(f"{self.config['temp_path']}/user_lookup")
        
        verbose_df = self.indexed_apriori \
            .join(user_lookup, on="userIndex", how="inner") \
            .join(self.item_lookup, on="itemIndex", how="inner")
            
        meta_subset = self.metadata_df.select("item_id", "title").dropDuplicates(["item_id"])
        final_verbose_df = verbose_df.join(meta_subset, on="item_id", how="left")
        
        return final_verbose_df.select("userId", "title", "total_score") \
            .orderBy(F.col("userId"), F.col("total_score").desc())

    def _generate_novel_recommendations(self, als_model, k):
        """
        Generates 2*k recommendations, filters out items the user has already seen 
        in the training data, and slices the array to return the top k novel items.
        """
        fetch_k = k * 2
        print(f"Generating top {fetch_k} raw recommendations to filter down to {k} novel items...")
        
        # 1. Get raw recommendations (Array of Structs)
        raw_recs = als_model.recommendForAllUsers(fetch_k)
        
        # 2. Aggregate the user's historical items into an array
        watched_items_df = self.indexed_train.groupBy("userIndex").agg(
            F.collect_set("itemIndex").alias("watched_items")
        )
        
        # 3. Join, filter natively using SQL expressions, and slice to top k
        novel_recs_df = raw_recs.join(watched_items_df, on="userIndex", how="left") \
            .withColumn("watched_items", F.coalesce(F.col("watched_items"), F.array().cast("array<int>"))) \
            .withColumn(
                "novel_recs_structs", 
                F.expr("filter(recommendations, x -> not array_contains(watched_items, x.itemIndex))")
            ) \
            .withColumn("top_k_structs", F.expr(f"slice(novel_recs_structs, 1, {k})")) \
            .select("userIndex", F.col("top_k_structs.itemIndex").alias("predicted_items"))
            
        return novel_recs_df

In [3]:
config = {
    "date": "2026-03-22",
    "train_days": 90,
    "test_days": 30,
    "watch_history_path": "gs://wynk-ml-workspace/projects/rails_reranking/daily_user_watch_history_new",
    "apriori_results_path": "gs://wynk-ml-workspace/xstream/clickApriori/",
    "db_path": "gs://wynk-ml-workspace/projects/xstream_nlu/catalog-db/",
    "temp_path": "gs://wynk-ml-workspace/_temp/harshith/als",
    "distinct_user_content_threshold": 2
}

pipeline = DataPipeline(spark, config)
pipeline.load_raw_data()
pipeline.process_and_index_data()

Loading raw interactions and metadata...


Path not found or inaccessible, skipping: gs://wynk-ml-workspace/projects/rails_reranking/daily_user_watch_history_new/day=2026-02-06
Path not found or inaccessible, skipping: gs://wynk-ml-workspace/projects/rails_reranking/daily_user_watch_history_new/day=2026-02-05


26/04/01 06:31:45 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Filtering users, aggregating playtime, and building indexes...


95th Percentile Threshold for max items watched: 41.0


Processing Apriori Recommendations...
Generating Ground Truth...


In [4]:
from pyspark.ml.recommendation import ALS
from pyspark.mllib.evaluation import RankingMetrics
from pyspark.sql.window import Window
import pyspark.sql.functions as F

def evaluate_models(pipeline, k=15, return_als=False, return_apriori=False):
    print(f"\n--- Fetching Data for Top {k} Evaluation ---")
    train_data, ground_truth = pipeline.get_als_data()
    apriori_data = pipeline.get_apriori_data()
    
    # ==========================================
    # 1. TRAIN ALS & GET TOP K
    # ==========================================
    print("Training ALS Model (Single Pass)...")
    als = ALS(
        userCol="userIndex", 
        itemCol="itemIndex", 
        ratingCol="playtime_logged", # Using the column generated in our pipeline
        implicitPrefs=True, 
        maxIter=10, 
        coldStartStrategy="drop",
        rank=100, 
        alpha=1.0
    )
    
    als_model = als.fit(train_data)
    
    # recommendForAllUsers returns an array of structs. We just need the item arrays.
    als_recs = als_model.recommendForAllUsers(k) 
    als_top_k = als_recs.select(
        "userIndex", 
        F.col("recommendations.itemIndex").alias("predicted_items")
    )

    # ==========================================
    # 2. FORMAT APRIORI TO MATCH ALS (TOP K)
    # ==========================================
    print("Formatting Apriori Recommendations...")
    # Rank items per user by total_score descending
    window_spec = Window.partitionBy("userIndex").orderBy(F.col("total_score").desc())
    
    apriori_top_k = apriori_data.withColumn("rank", F.row_number().over(window_spec)) \
        .filter(F.col("rank") <= k) \
        .groupBy("userIndex") \
        .agg(F.collect_list("itemIndex").alias("predicted_items"))

    # ==========================================
    # 3. EVALUATION HELPER FUNCTION
    # ==========================================
    def print_metrics(model_name, predictions_df):
        # Join predictions with ground truth
        joined_data = predictions_df.join(ground_truth, "userIndex").dropna()
        
        # DataFrame calculation for Recall (since PySpark RankingMetrics lacks recallAt in Python)
        eval_df = joined_data \
            .withColumn("hits", F.size(F.array_intersect(F.col("predicted_items"), F.col("actual_items")))) \
            .withColumn("actual_count", F.size(F.col("actual_items"))) \
            .withColumn("recall", F.col("hits") / F.col("actual_count"))
            
        avg_recall = eval_df.select(F.mean("recall")).collect()[0][0]
        
        # RDD conversion for PySpark's native Precision and MAP
        rdd_data = joined_data.select("predicted_items", "actual_items").rdd.map(tuple)
        metrics = RankingMetrics(rdd_data)
        
        print(f"\n[{model_name} Performance]")
        print(f"Users Evaluated: {joined_data.count()}")
        print(f"Precision@{k}:    {metrics.precisionAt(k):.4f}")
        print(f"Recall@{k}:       {avg_recall:.4f}")
        print(f"MAP:             {metrics.meanAveragePrecision:.4f}")

    # ==========================================
    # 4. RUN THE METRICS
    # ==========================================
    print("\nCalculating Metrics...")
    print_metrics("ALS Model", als_top_k)
    print_metrics("Apriori Model", apriori_top_k)

    if return_als:
        return als_model

# How to run it:
# evaluate_models_head_to_head(pipeline, k=15)

In [5]:
trained_als_model = evaluate_models(pipeline, k=15, return_als = True)


--- Fetching Data for Top 15 Evaluation ---
Training ALS Model (Single Pass)...


26/04/01 06:33:07 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/04/01 06:33:07 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/04/01 06:33:35 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 23 for reason Executor for container container_1764236692086_6088_01_000025 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.


Formatting Apriori Recommendations...

Calculating Metrics...


26/04/01 06:40:19 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/04/01 06:40:19 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/04/01 06:40:25 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/04/01 06:40:36 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/04/01 06:


[ALS Model Performance]


...


[Apriori Model Performance]


26/04/01 06:44:29 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


Users Evaluated: 2463027


Precision@15:    0.0331
Recall@15:       0.1084


MAP:             0.0387


In [25]:
import pyspark.sql.functions as F

class ALSAnalyzer:
    def __init__(self, spark, trained_als_model, pipeline):
        self.spark = spark
        self.model = trained_als_model
        self.pipeline = pipeline
        
        # Cache factor matrices
        self.item_factors = self.model.itemFactors.cache()
        self.user_factors = self.model.userFactors.cache()

    def _calculate_cosine_similarity(self, factors_df, target_index, top_n):
        """Internal helper for native array vector math."""
        target_row = factors_df.filter(F.col("id") == target_index).select("features").first()
        if not target_row:
            raise ValueError(f"Internal Index {target_index} not found in the trained model.")
        
        target_vector = target_row["features"]
        target_col = F.array([F.lit(x) for x in target_vector])
        
        dot_product_expr = F.expr(
            "aggregate(zip_with(features, target, (x, y) -> x * y), 0D, (acc, x) -> acc + x)"
        )
        norm_features_expr = F.expr(
            "sqrt(aggregate(features, 0D, (acc, x) -> acc + (x * x)))"
        )
        norm_target_expr = F.expr(
            "sqrt(aggregate(target, 0D, (acc, x) -> acc + (x * x)))"
        )
        
        similarity_df = factors_df.withColumn("target", target_col) \
            .withColumn("dot_product", dot_product_expr) \
            .withColumn("norm_features", norm_features_expr) \
            .withColumn("norm_target", norm_target_expr) \
            .withColumn("cosine_similarity", F.col("dot_product") / (F.col("norm_features") * F.col("norm_target")))
            
        return similarity_df.filter(F.col("id") != target_index) \
            .select("id", "cosine_similarity") \
            .orderBy(F.col("cosine_similarity").desc()) \
            .limit(top_n)

    def find_closest_items(self, item_id, top_n=5):
        """Finds closest items using a string item_id and prints metadata first."""
        # 1. Look up the itemIndex
        item_row = self.pipeline.item_lookup.filter(F.col("item_id") == item_id).select("itemIndex").first()
        if not item_row:
            raise ValueError(f"Item ID '{item_id}' not found in the lookup table.")
        target_item_index = item_row["itemIndex"]

        # 2. Print Target Item Information
        meta_row = self.pipeline.metadata_df.filter(F.col("item_id") == item_id).first()
        print("-" * 50)
        if meta_row:
            print(f"TARGET ITEM INFO:\nID: {item_id}\nTitle: {meta_row['title']}\nGenres: {meta_row['Genres']}")
        else:
            print(f"TARGET ITEM INFO:\nID: {item_id}\n(Metadata not found)")
        print("-" * 50)
        
        # 3. Calculate Similarity
        print(f"Finding top {top_n} closest matches...")
        closest_df = self._calculate_cosine_similarity(self.item_factors, target_item_index, top_n)
        
        # 4. Join back to metadata for readability
        readable_df = closest_df.withColumnRenamed("id", "itemIndex") \
            .join(self.pipeline.item_lookup, "itemIndex", "inner") \
            .join(self.pipeline.metadata_df, "item_id", "left") \
            .select("item_id", "title", "Genres", "cosine_similarity") \
            .orderBy(F.col("cosine_similarity").desc())
            
        return readable_df

    def recommend_for_user(self, userId, top_n=10, extra_history_ids=None):
        """
        Gets history from train data for a userId, prints it, and generates recommendations.
        Also accepts 'extra_history_ids' if it's a completely cold user with live session data.
        """
        print("-" * 50)
        print(f"PROCESSING USER: {userId}")
        
        history_item_ids = set(extra_history_ids) if extra_history_ids else set()

        # 1. Check if user is in training data and extract history
        user_row = self.pipeline.user_lookup.filter(F.col("userId") == userId).select("userIndex").first()
        
        if user_row:
            user_index = user_row["userIndex"]
            print(f"User found in training data (Index: {user_index}). Extracting history...")
            
            # Fetch history directly from indexed_train
            history_df = self.pipeline.indexed_train.filter(F.col("userIndex") == user_index)
            history_indices = [row["itemIndex"] for row in history_df.select("itemIndex").collect()]
            
            # Translate indices back to item_ids
            if history_indices:
                item_id_df = self.pipeline.item_lookup.filter(F.col("itemIndex").isin(history_indices))
                train_history_ids = [row["item_id"] for row in item_id_df.select("item_id").collect()]
                history_item_ids.update(train_history_ids)
        else:
            print("User NOT found in training data (Cold Start).")

        history_list = list(history_item_ids)
        print(f"Total Items in History to Vectorize: {len(history_list)}")
        print("-" * 50)

        if not history_list:
            print("No watch history available to generate recommendations.")
            return None

        # 2. Pass the aggregated history to generate recommendations
        return self._generate_recommendations_from_history(history_list, top_n)

    def _generate_recommendations_from_history(self, item_ids, top_n=10):
        """Generates recommendations by aggregating vectors of provided item_ids."""
        # Translate item_ids to itemIndex
        lookup_df = self.pipeline.item_lookup.filter(F.col("item_id").isin(item_ids))
        watched_item_indices = [row["itemIndex"] for row in lookup_df.collect()]
        
        if not watched_item_indices:
             raise ValueError("None of the provided history item IDs exist in the trained model.")
             
        # Retrieve Item Vectors for the history
        history_df = self.item_factors.filter(F.col("id").isin(watched_item_indices))
        history_vectors = [row["features"] for row in history_df.collect()]
        
        # Aggregate (Average) the Vectors
        num_items = len(history_vectors)
        vector_length = len(history_vectors[0])
        synthetic_vector = [sum(vec[i] for vec in history_vectors) / num_items for i in range(vector_length)]
        
        synth_col = F.array([F.lit(x) for x in synthetic_vector])
        
        # Filter out items they've already watched
        candidates_df = self.item_factors.filter(~F.col("id").isin(watched_item_indices))
        
        # Calculate Standard ALS Score (Dot Product)
        dot_product_expr = F.expr(
            "aggregate(zip_with(features, synthetic_user, (x, y) -> x * y), 0D, (acc, x) -> acc + x)"
        )
        
        recommendations = candidates_df.withColumn("synthetic_user", synth_col) \
            .withColumn("recommendation_score", dot_product_expr) \
            .select("id", "recommendation_score") \
            .orderBy(F.col("recommendation_score").desc()) \
            .limit(top_n)
            
        # Join back to metadata for readability
        readable_recs = recommendations.withColumnRenamed("id", "itemIndex") \
            .join(self.pipeline.item_lookup, "itemIndex", "inner") \
            .join(self.pipeline.metadata_df, "item_id", "left") \
            .select("item_id", "title", "recommendation_score") \
            .orderBy(F.col("recommendation_score").desc())
            
        return readable_recs
    def _get_synthetic_vector(self, userId):
        """Helper: Extracts raw history from train_df and builds the synthetic vector."""
        # 1. Pull history from the RAW train_df to ensure we catch dropped users
        history_df = self.pipeline.train_df.filter(F.col("userId") == userId).select("item_id").distinct()
        
        # 2. Map their history to our trained item indices
        history_indices_df = history_df.join(self.pipeline.item_lookup, "item_id", "inner")
        watched_item_indices = [row["itemIndex"] for row in history_indices_df.collect()]
        
        if not watched_item_indices:
            print(f"User {userId} has no valid history in the training data.")
            return None, []
            
        # 3. Fetch the item vectors for what they watched
        history_vectors_df = self.item_factors.filter(F.col("id").isin(watched_item_indices))
        history_vectors = [row["features"] for row in history_vectors_df.collect()]
        
        if not history_vectors:
             print(f"None of the items User {userId} watched made it into the trained model.")
             return None, watched_item_indices
             
        # 4. Calculate the mathematical average to create the Synthetic User Vector
        num_items = len(history_vectors)
        vector_length = len(history_vectors[0])
        synthetic_vector = [sum(vec[i] for vec in history_vectors) / num_items for i in range(vector_length)]
        
        return synthetic_vector, watched_item_indices

    def recommend_via_synthetic_vector(self, userId, top_n=10):
        """CURRENT APPROACH: Computes S * I for direct item recommendations."""
        print("-" * 50)
        print(f"GENERATING DIRECT RECOMMENDATIONS FOR: {userId}")
        
        synthetic_vector, watched_item_indices = self._get_synthetic_vector(userId)
        if not synthetic_vector:
            return None
            
        print(f"Successfully built synthetic vector from {len(watched_item_indices)} watched items.")
        print("-" * 50)
        
        # Convert vector to Spark Column
        synth_col = F.array([F.lit(x) for x in synthetic_vector])
        
        # Filter out what they already watched
        candidates_df = self.item_factors.filter(~F.col("id").isin(watched_item_indices))
        
        # Compute Dot Product (Score = S * I)
        dot_product_expr = F.expr(
            "aggregate(zip_with(features, synthetic_user, (x, y) -> x * y), 0D, (acc, x) -> acc + x)"
        )
        
        recommendations = candidates_df.withColumn("synthetic_user", synth_col) \
            .withColumn("recommendation_score", dot_product_expr) \
            .orderBy(F.col("recommendation_score").desc()) \
            .limit(top_n)
            
        # Join back to metadata
        return recommendations.withColumnRenamed("id", "itemIndex") \
            .join(self.pipeline.item_lookup, "itemIndex", "inner") \
            .join(self.pipeline.metadata_df, "item_id", "left") \
            .select("item_id", "title", "Genres", "recommendation_score") \
            .orderBy(F.col("recommendation_score").desc())

    def find_taste_neighbors(self, userId, top_n=5):
        """FUTURE TODO: Computes Cosine Similarity between Synthetic Vector and User Factors."""
        print("-" * 50)
        print(f"FINDING TASTE NEIGHBORS FOR: {userId}")
        
        synthetic_vector, _ = self._get_synthetic_vector(userId)
        if not synthetic_vector:
            return None
            
        # Convert vector to Spark Column
        target_col = F.array([F.lit(x) for x in synthetic_vector])
        
        # Compute Cosine Similarity (S vs U)
        dot_product_expr = F.expr(
            "aggregate(zip_with(features, target, (x, y) -> x * y), 0D, (acc, x) -> acc + x)"
        )
        norm_features_expr = F.expr(
            "sqrt(aggregate(features, 0D, (acc, x) -> acc + (x * x)))"
        )
        # Norm of synthetic vector is a constant, compute it in python to save cluster work
        import math
        norm_target_val = math.sqrt(sum(x*x for x in synthetic_vector))
        
        similarity_df = self.user_factors.withColumn("target", target_col) \
            .withColumn("dot_product", dot_product_expr) \
            .withColumn("norm_features", norm_features_expr) \
            .withColumn("cosine_similarity", F.col("dot_product") / (F.col("norm_features") * F.lit(norm_target_val)))
            
        # Exclude the user themselves if they happened to be in the training set
        user_row = self.pipeline.user_lookup.filter(F.col("userId") == userId).select("userIndex").first()
        if user_row:
            similarity_df = similarity_df.filter(F.col("id") != user_row["userIndex"])
            
        neighbors = similarity_df.orderBy(F.col("cosine_similarity").desc()).limit(top_n)
        
        return neighbors.withColumnRenamed("id", "userIndex") \
            .join(self.pipeline.user_lookup, "userIndex", "inner") \
            .select("userId", "cosine_similarity") \
            .orderBy(F.col("cosine_similarity").desc())

In [26]:
# 1. Train your model using the data from your pipeline
train_data, ground_truth = pipeline.get_als_data()
from pyspark.ml.recommendation import ALS
als = ALS(userCol="userIndex", itemCol="itemIndex", ratingCol="playtime_logged", rank=50)
model = als.fit(train_data)

# 2. Initialize the Analyzer
analyzer = ALSAnalyzer(spark, model, pipeline)

# 1. Current Goal: Direct Recommendations via Synthetic Vector
direct_recs = analyzer.recommend_via_synthetic_vector(userId="ib-xPv06AjuYOE4BI0", top_n=10)
if direct_recs:
    direct_recs.show(truncate=False)

# 2. Future Goal: User Clustering / Taste Neighbors
neighbors = analyzer.find_taste_neighbors(userId="ib-xPv06AjuYOE4BI0", top_n=5)
if neighbors:
    neighbors.show(truncate=False)

26/03/31 10:39:23 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 50 for reason Executor for container container_1764236692086_6041_01_000099 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.


--------------------------------------------------
GENERATING DIRECT RECOMMENDATIONS FOR: ib-xPv06AjuYOE4BI0


26/03/31 10:40:29 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


Successfully built synthetic vector from 2 watched items.
--------------------------------------------------


26/03/31 10:40:31 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/31 10:40:32 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


+---------------------------------------------------------------+---------------------+---------------------------+--------------------+
|item_id                                                        |title                |Genres                     |recommendation_score|
+---------------------------------------------------------------+---------------------+---------------------------+--------------------+
|MINITV_TVSHOW_amzn1.dv.gti.96557ab3-0e41-4b02-8f30-010e081f83e7|NULL                 |NULL                       |5.808496205786772   |
|SHEMAROOME_MOVIE_5ba8cadac1df413dae00024b                      |Kaajal               |[Music, Romance]           |5.68036523259262    |
|ZEEFIVE_MOVIE_0-0-1z5273898                                    |Thothapuri: Chapter 1|[Comedy]                   |5.327277512297027   |
|EROSNOW_MOVIE_6832603                                          |Sniff!!!             |[Adventure, Comedy, Family]|5.251568276255089   |
|ZEEFIVE_MOVIE_0-0-1z5214566             

26/03/31 10:40:32 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


AttributeError: 'DataPipeline' object has no attribute 'user_lookup'

26/03/31 10:44:26 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 11 for reason Executor for container container_1764236692086_6041_01_000028 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/31 10:44:29 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 10 for reason Executor for container container_1764236692086_6041_01_000026 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
